<a href="https://colab.research.google.com/github/higgsboson1710/CXR-MultiQuant./blob/main/notebooks/ModelArchitecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CXR-MultiQuant — Phase 2: Model Architecture
Building the Multimodal deep learning architecture using **TensorFlow / Keras**.

- **Vision**: DenseNet-121 (ImageNet pretrained)
- **Text**: BERT-base via TensorFlow Hub (same architecture as ClinicalBERT: 12 layers, 768 hidden)
- **Fusion**: Co-Attention (2× Multi-Head Attention)
- **Loss**: Focal Loss (handles class imbalance)

In [12]:
# Cell 1: Install Dependencies
# transformers 5.x removed TF support. We MUST pin to 4.37.2 to use ClinicalBERT.
!pip install --quiet "transformers==4.37.2" tf-keras datasets pandas scikit-learn keras-tuner


In [13]:
# Cell 2: Imports and Setup
import os
import io

# CRITICAL: Force HuggingFace transformers to use tf-keras (Legacy Keras 2)
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from transformers import AutoTokenizer, TFAutoModel  # Required for ClinicalBERT

from datasets import load_dataset
from PIL import Image

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow Version: {tf.__version__}")
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("All imports successful!")


TensorFlow Version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
All imports successful!


In [14]:
# Cell 6: Text Encoder (ClinicalBERT via HuggingFace)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
MAX_LEN = 512

def create_text_encoder():


    input_ids = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
    attention_mask = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")

    # CRITICAL FIX for TensorFlow 2.16+ Keras 3 backend compatibility with HuggingFace
    import keras.backend as K
    if not hasattr(K, 'set_value'):
        K.set_value = lambda x, y: x.assign(y) if hasattr(x, 'assign') else None

    # Load Hugging Face TF ClinicalBERT
    clinical_bert = TFAutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT", from_pt=True)
    clinical_bert.trainable = False  # Freeze for initial training

    bert_output = clinical_bert(input_ids, attention_mask=attention_mask)

    # Extract [CLS] Token (first token in sequence)
    cls_token = bert_output.last_hidden_state[:, 0, :]

    # Dense Projection Layer -> 256-dim
    txt_features = layers.Dense(256, activation='relu', name="txt_projection")(cls_token)

    return keras.Model(inputs=[input_ids, attention_mask], outputs=txt_features, name="Text_Encoder")

print("Downloading ClinicalBERT model from Hugging Face...")
text_encoder = create_text_encoder()
text_encoder.summary()


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already

Model: "Text_Encoder"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_ids (InputLayer)      [(None, 512)]                0         []                            
                                                                                                  
 attention_mask (InputLayer  [(None, 512)]                0         []                            
 )                                                                                                
                                                                                                  
 tf_bert_model_3 (TFBertMod  TFBaseModelOutputWithPooli   1083102   ['input_ids[0][0]',           
 el)                         ngAndCrossAttentions(last_   72         'attention_mask[0][0]']      
                             hidden_state=(None, 512, 7                                

In [15]:
# Cell 4: Create tf.data.Dataset Generator

label_map = {'Mild': 0, 'Moderate': 1, 'Severe': 2}

def extract_image(img_data):
    if isinstance(img_data, dict) and 'bytes' in img_data:
        img = Image.open(io.BytesIO(img_data['bytes'])).convert('RGB')
    else:
        img = img_data.convert('RGB')

    img = img.resize((224, 224))
    return np.array(img) / 255.0  # Normalize to [0, 1]

def data_generator(df):
    for idx, row in df.iterrows():
        img_array = extract_image(row['image'])

        tokens = tokenizer(
            str(row['findings']) + " " + str(row['impression']),
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors="np"  # Return numpy arrays (compatible with tf.data)
        )
        input_ids = tokens['input_ids'].squeeze()
        attention_mask = tokens['attention_mask'].squeeze()

        label = label_map[row['severity']]
        yield (img_array, input_ids, attention_mask), label

output_signature = (
    (
        tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32),  # Image
        tf.TensorSpec(shape=(MAX_LEN,), dtype=tf.int32),       # Input IDs
        tf.TensorSpec(shape=(MAX_LEN,), dtype=tf.int32)        # Attention Mask
    ),
    tf.TensorSpec(shape=(), dtype=tf.int32)                    # Label
)

BATCH_SIZE = 16



train_ds = tf.data.Dataset.from_generator(lambda: data_generator(train_df), output_signature=output_signature)
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(lambda: data_generator(val_df), output_signature=output_signature)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("tf.data Pipelines created successfully!")

tf.data Pipelines created successfully!


## 1. Image Encoder (DenseNet-121)
Pretrained on ImageNet, augmented, and projected to 256 dimensions.

In [16]:
# Cell 5: Image Encoder
def create_image_encoder():
    image_input = keras.Input(shape=(224, 224, 3), name="image_input")

    # Data Augmentation
    x = layers.RandomRotation(0.1)(image_input)
    x = layers.RandomZoom(0.1)(x)

    # Pretrained DenseNet121
    densenet = keras.applications.DenseNet121(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )
    densenet.trainable = False  # Freeze for initial training

    x = densenet.output
    x = layers.GlobalAveragePooling2D()(x)

    # Dense Projection Layer -> 256-dim
    img_features = layers.Dense(256, activation='relu', name="img_projection")(x)

    return keras.Model(inputs=image_input, outputs=img_features, name="Image_Encoder")

image_encoder = create_image_encoder()
image_encoder.summary()

29084464/29084464 [==============================] - 0s 0us/step
Model: "Image_Encoder"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 image_input (InputLayer)    [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 random_rotation (RandomRot  (None, 224, 224, 3)          0         ['image_input[0][0]']         
 ation)                                                                                           
                                                                                                  
 random_zoom (RandomZoom)    (None, 224, 224, 3)          0         ['random_rotation[0][0]']     
                                                                                                  
 zero_padding2d (Zero

## 2. Text Encoder (ClinicalBERT via HuggingFace)
Uses Bio_ClinicalBERT (12 layers, 768 hidden dims, 12 attention heads).
Extracts the [CLS] pooled output and projects to 256 dimensions.


In [17]:
# Cell 6: Text Encoder (ClinicalBERT via HuggingFace)
def create_text_encoder():
    input_ids = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
    attention_mask = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")

    # CRITICAL FIX for TensorFlow 2.16+ Keras 3 backend compatibility with HuggingFace
    import keras.backend as K
    if not hasattr(K, 'set_value'):
        K.set_value = lambda x, y: x.assign(y) if hasattr(x, 'assign') else None

    # Load Hugging Face TF ClinicalBERT
    clinical_bert = TFAutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT", from_pt=True)
    clinical_bert.trainable = False  # Freeze for initial training

    bert_output = clinical_bert(input_ids, attention_mask=attention_mask)

    # Extract [CLS] Token (first token in sequence)
    cls_token = bert_output.last_hidden_state[:, 0, :]

    # Dense Projection Layer -> 256-dim
    txt_features = layers.Dense(256, activation='relu', name="txt_projection")(cls_token)

    return keras.Model(inputs=[input_ids, attention_mask], outputs=txt_features, name="Text_Encoder")

print("Downloading ClinicalBERT model from Hugging Face...")
text_encoder = create_text_encoder()
text_encoder.summary()


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already

Model: "Text_Encoder"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_ids (InputLayer)      [(None, 512)]                0         []                            
                                                                                                  
 attention_mask (InputLayer  [(None, 512)]                0         []                            
 )                                                                                                
                                                                                                  
 tf_bert_model_4 (TFBertMod  TFBaseModelOutputWithPooli   1083102   ['input_ids[0][0]',           
 el)                         ngAndCrossAttentions(last_   72         'attention_mask[0][0]']      
                             hidden_state=(None, 512, 7                                

## 3. Co-Attention Fusion & Classifier
Cross attention between Image and Text, followed by the severity classifier.

In [18]:
# Cell 7: Full Multimodal Model with Co-Attention
def create_multimodal_model():
    # Inputs
    image_input = keras.Input(shape=(224, 224, 3), name="image_input")
    input_ids = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
    attention_mask = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")

    # Encoders
    img_feats = image_encoder(image_input)                 # (batch, 256)
    txt_feats = text_encoder([input_ids, attention_mask])  # (batch, 256)

    # To use MultiHeadAttention, we need 3D tensors: (batch_size, seq_len, dim)
    img_feats_seq = tf.expand_dims(img_feats, axis=1)      # (batch, 1, 256)
    txt_feats_seq = tf.expand_dims(txt_feats, axis=1)      # (batch, 1, 256)

    # --- CO-ATTENTION FUSION ---
    # Cross 1: Image -> Text (Q: Image, K/V: Text)
    attn_img_txt = layers.MultiHeadAttention(num_heads=8, key_dim=32, name="cross1_img_txt")(
        query=img_feats_seq, value=txt_feats_seq, key=txt_feats_seq
    )
    # Residual & LayerNorm
    norm1 = layers.LayerNormalization()(img_feats_seq + attn_img_txt)

    # Cross 2: Text -> Image (Q: Text, K/V: Image)
    attn_txt_img = layers.MultiHeadAttention(num_heads=8, key_dim=32, name="cross2_txt_img")(
        query=txt_feats_seq, value=img_feats_seq, key=img_feats_seq
    )
    # Residual & LayerNorm
    norm2 = layers.LayerNormalization()(txt_feats_seq + attn_txt_img)

    # Squeeze back to 2D and Concatenate -> 512-dim
    norm1_flat = tf.squeeze(norm1, axis=1)
    norm2_flat = tf.squeeze(norm2, axis=1)
    fused = layers.Concatenate(name="fused_concat")([norm1_flat, norm2_flat])

    # --- CLASSIFIER HEAD ---
    x = layers.Dense(256, activation='relu', name="fc1")(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu', name="fc2")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    output = layers.Dense(3, activation='softmax', name="severity_prediction")(x)

    return keras.Model(inputs=[image_input, input_ids, attention_mask], outputs=output, name="CXR_MultiQuant")

multimodal_model = create_multimodal_model()
multimodal_model.summary()

Model: "CXR_MultiQuant"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 image_input (InputLayer)    [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 input_ids (InputLayer)      [(None, 512)]                0         []                            
                                                                                                  
 attention_mask (InputLayer  [(None, 512)]                0         []                            
 )                                                                                                
                                                                                                  
 Image_Encoder (Functional)  (None, 256)                  7299904   ['image_input[0][

## 4. Compile with Focal Loss

In [19]:
# Cell 8: Define Focal Loss and Compile Model
class CategoricalFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        # y_true is passed as integer labels from the dataset (sparse)
        # We need to one-hot encode them first
        y_true_one_hot = tf.one_hot(tf.cast(y_true, tf.int32), depth=3)

        y_pred = tf.clip_by_value(y_pred, keras.backend.epsilon(), 1.0 - keras.backend.epsilon())
        cross_entropy = -y_true_one_hot * tf.math.log(y_pred)
        loss = self.alpha * tf.math.pow(1 - y_pred, self.gamma) * cross_entropy
        return tf.reduce_sum(loss, axis=-1)

multimodal_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=CategoricalFocalLoss(gamma=2.0, alpha=0.25),
    metrics=['accuracy', keras.metrics.AUC(name='auc_roc', multi_label=True)]
)

print("Model compiled successfully with Focal Loss!")

Model compiled successfully with Focal Loss!


In [20]:
# Cell 9: Hyperparameter Tuning (Bayesian Optimization)
import keras_tuner as kt
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

def build_model(hp):
    # Rebuild the model structure for the tuner
    image_input = keras.Input(shape=(224, 224, 3), name="image_input")
    input_ids = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
    attention_mask = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")

    img_feats = image_encoder(image_input)
    txt_feats = text_encoder([input_ids, attention_mask])

    img_feats_seq = tf.expand_dims(img_feats, axis=1)
    txt_feats_seq = tf.expand_dims(txt_feats, axis=1)

    attn_img_txt = layers.MultiHeadAttention(num_heads=8, key_dim=32)(
        query=img_feats_seq, value=txt_feats_seq, key=txt_feats_seq
    )
    norm1 = layers.LayerNormalization()(img_feats_seq + attn_img_txt)

    attn_txt_img = layers.MultiHeadAttention(num_heads=8, key_dim=32)(
        query=txt_feats_seq, value=img_feats_seq, key=img_feats_seq
    )
    norm2 = layers.LayerNormalization()(txt_feats_seq + attn_txt_img)

    norm1_flat = tf.squeeze(norm1, axis=1)
    norm2_flat = tf.squeeze(norm2, axis=1)
    fused = layers.Concatenate()([norm1_flat, norm2_flat])

    # Hyperparameters for Dense and Dropout
    hp_dropout_1 = hp.Choice('dropout_1', values=[0.2, 0.3, 0.5])
    hp_dropout_2 = hp.Choice('dropout_2', values=[0.2, 0.3, 0.5])

    x = layers.Dense(256, activation='relu')(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(hp_dropout_1)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(hp_dropout_2)(x)

    output = layers.Dense(3, activation='softmax')(x)

    model = keras.Model(inputs=[image_input, input_ids, attention_mask], outputs=output)

    # Hyperparameters for Optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-3, 1e-4, 1e-5])
    hp_optimizer = hp.Choice('optimizer', values=['adam', 'rmsprop'])

    if hp_optimizer == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)
    else:
        optimizer = keras.optimizers.RMSprop(learning_rate=hp_learning_rate)

    model.compile(
        optimizer=optimizer,
        loss=CategoricalFocalLoss(gamma=2.0, alpha=0.25),
        metrics=['accuracy', keras.metrics.AUC(name='auc_roc', multi_label=True)]
    )
    return model

tuner = kt.BayesianOptimization(
    build_model,
    objective=kt.Objective('val_auc_roc', direction='max'),
    max_trials=10,  # Bayesian Optimization trials
    directory='keras_tuner_dir',
    project_name='CXR_MultiQuant_Tuning'
)

tuner.search_space_summary()


Search space summary
Default search space size: 4
dropout_1 (Choice)
{'default': 0.2, 'conditions': [], 'values': [0.2, 0.3, 0.5], 'ordered': True}
dropout_2 (Choice)
{'default': 0.2, 'conditions': [], 'values': [0.2, 0.3, 0.5], 'ordered': True}
learning_rate (Choice)
{'default': 0.001, 'conditions': [], 'values': [0.001, 0.0001, 1e-05], 'ordered': True}
optimizer (Choice)
{'default': 'adam', 'conditions': [], 'values': ['adam', 'rmsprop'], 'ordered': False}


In [21]:
# Cell 10: Callbacks & Start Tuning
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_loss'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

# To start tuning, uncomment the line below:
tuner.search(train_ds, validation_data=val_ds, epochs=30, callbacks=callbacks)

# To retrieve the best model after tuning:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()



Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
0.5               |0.5               |dropout_1
0.2               |0.2               |dropout_2
0.001             |0.001             |learning_rate
adam              |adam              |optimizer



FatalTypeError: Expected the model-building function, or HyperModel.build() to return a valid Keras Model instance. Received: <tf_keras.src.engine.functional.Functional object at 0x7ddf5e753c80> of type <class 'tf_keras.src.engine.functional.Functional'>.